# Same AUC, hidden leak — a 5-minute tour of `dextra`

Two preprocessing protocols. One fits its statistics on **all** rows and splits
afterwards; the other splits first, fits on **train only**, and replays a frozen
`params` artifact on test. A single run scores them almost identically —
repeated splits expose the first as systematically optimistic.

`dextra` makes the safe protocol the path of least resistance: every stateful
step returns a replayable, JSON-serialisable `params` plan.

*Runs top-to-bottom on a fresh Colab runtime in about 5 minutes. Data: IBM's*
*public Telco customer-churn sample (7,043 rows).*


In [ ]:
%pip install -q pydextra scikit-learn


## Load — with full disclosure

`dx.load` turns the raw CSV into a typed DataFrame and *tells you everything it
did*: encoding, delimiter, every coerced column, and a one-line `Decision:`.


In [ ]:
import urllib.request

import pandas as pd
import dextra as dx

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
urllib.request.urlretrieve(URL, "Telco-Customer-Churn.csv")

data = dx.load("Telco-Customer-Churn.csv")
data = data.drop(columns=["customerID"], errors="ignore")


## Three stateful steps

Each of these *learns numbers* from whatever data you feed it — a median,
one-hot vocabularies, per-column means and stds. Feed it the test rows and
those numbers silently carry test information into training.


In [ ]:
CAT = ["gender", "MultipleLines", "InternetService", "OnlineSecurity",
       "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
       "StreamingMovies", "Contract", "PaymentMethod"]
NUM = ["tenure", "MonthlyCharges", "TotalCharges"]

TELCO_STEPS = [
    {"fn": "handle_missing", "strategy": {"TotalCharges": "median"}},   # learns a median
    {"fn": "encode", "cols": CAT, "method": "onehot", "inplace": True}, # learns vocabularies
    {"fn": "scale", "cols": NUM, "method": "standard", "inplace": True},# learns mean/std
]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


def test_auc(train_fe, test_fe, target="Churn"):
    """One fixed measuring device for both protocols."""
    X_tr = train_fe.drop(columns=[target])
    X_te = test_fe.drop(columns=[target]).reindex(columns=X_tr.columns, fill_value=0)
    model = LogisticRegression(max_iter=2000).fit(X_tr, train_fe[target].astype(int))
    return roc_auc_score(test_fe[target].astype(int), model.predict_proba(X_te)[:, 1])


train_idx, test_idx = train_test_split(
    data.index, test_size=0.25, random_state=42, stratify=data["Churn"])
print(f"train {len(train_idx):,} rows | test {len(test_idx):,} rows")


## ❌ Protocol A — fit on ALL rows, split afterwards

This is what most tutorial code does without noticing.


In [ ]:
fe_all = dx.featpipe(data, steps=TELCO_STEPS, show=False, plot=False)
auc_wrong = test_auc(fe_all.loc[train_idx], fe_all.loc[test_idx])
print(f"leaky protocol   test AUC = {auc_wrong:.4f}")


## ✅ Protocol B — split first, fit on train, replay frozen `params`

FIT mode prints its full report — watch it disclose every fitted statistic,
ending in the `Decision:` line. The returned `params` is a versioned,
JSON-serialisable artifact: commit it, ship it, replay it.


In [ ]:
train_fe, telco_params = dx.featpipe(
    data.loc[train_idx], steps=TELCO_STEPS, return_params=True)
test_fe = dx.featpipe(data.loc[test_idx], params=telco_params, show=False, plot=False)
auc_right = test_auc(train_fe, test_fe)
print(f"clean protocol   test AUC = {auc_right:.4f}")


## One run tells you nothing


In [ ]:
print(f"leaky protocol : {auc_wrong:.4f}")
print(f"clean protocol : {auc_right:.4f}")
print(f"difference     : {auc_wrong - auc_right:+.4f}")


Nearly identical numbers. If you stopped here you would ship the leaky
pipeline with a clear conscience. So we call a referee.


## The referee — the same experiment on 10 fresh splits


In [ ]:
def protocol_auc(seed, leaky):
    """One full run of either protocol on a fresh random split."""
    tr_i, te_i = train_test_split(data.index, test_size=0.25,
                                  random_state=seed, stratify=data["Churn"])
    if leaky:   # fit preprocessing on ALL rows, then split
        fe = dx.featpipe(data, steps=TELCO_STEPS, show=False, plot=False)
        tr, te = fe.loc[tr_i], fe.loc[te_i]
    else:       # split first, fit on train only, APPLY frozen params to test
        tr, p = dx.featpipe(data.loc[tr_i], steps=TELCO_STEPS,
                            return_params=True, show=False, plot=False)
        te = dx.featpipe(data.loc[te_i], params=p, show=False, plot=False)
    return test_auc(tr, te)


SEEDS = range(10)
results = pd.DataFrame({
    "leaky": [protocol_auc(s, leaky=True) for s in SEEDS],
    "clean": [protocol_auc(s, leaky=False) for s in SEEDS],
}, index=pd.Index(SEEDS, name="seed"))

bias = results["leaky"] - results["clean"]
print(results.round(4))
print(f"\nmean optimism of the leaky protocol : {bias.mean():+.4f} ± {bias.std():.4f}")
print(f"leaky scored higher on {(bias > 0).sum()} of {len(bias)} splits")


## Takeaway

- **One split**: the two protocols look interchangeable.
- **Ten splits**: the leaky one is systematically optimistic — a bias you
  would have shipped as "model performance".
- With `dextra`, the safe protocol is *less* code, not more: fit once with
  `return_params=True`, replay with `params=`. The artifact is versioned
  JSON — it goes in your repo next to the model.

**Go deeper:** the full story — including how feature selection amplifies the
leak — is in
[notebook 02](https://github.com/ahmedabdeltawab602-collab/dextra/blob/main/notebooks/02-leakage-safe-pipeline.ipynb).

[Documentation](https://ahmedabdeltawab602-collab.github.io/dextra/) ·
[GitHub](https://github.com/ahmedabdeltawab602-collab/dextra) ·
[PyPI](https://pypi.org/project/pydextra/)

*`pydextra`'s public API is frozen since 0.6.0 — what you evaluated today is*
*what you run next year. Bug reports are read and answered.*

*Data: IBM Telco customer-churn public sample. Library: MIT license.*
